# Compare two inference-server images

This notebook shows the results of the pattern's image comparison: two server images, the same model, the same GPU and the same requests, measured in rounds that take turns between them (A B, B A, A B, ...). The comparison runs as a Job, not in this notebook; it is switched on with global.compare in values-global.yaml, and every run writes one results.json to the results volume.

Run the cells from the top. Each cell reads the newest run; pass a run id, for example show.verdicts("my-run-1a2b3c4d"), to see an older one. To check the layout without a cluster, start JupyterLab with RHVP_COMPARE_REPLAY set to compare-recordings/synthetic.json.

In [ ]:
from rhvp_compare import show

show.runs()

## 1 · Verdicts

One row per metric. B/A is the median, over the pairs, of side B's figure divided by side A's in the same pair; the interval next to it is a distribution-free confidence interval for that median. The verdict follows a rule fixed before the first round: TIE when the median and its whole interval lie inside the tie band; outside the band, B is FASTER or SLOWER (MORE_EFFICIENT or LESS_EFFICIENT for tokens per joule) only when the p-value is at most alpha and the test leans the same way as the median; otherwise INCONCLUSIVE. A metric is also INCONCLUSIVE when one side failed in some pair on its own account, or when the rounds ran on more than one node. Only the primary metrics, declared before the first round, count as findings: their p-values are corrected for the number of primary metrics. The secondary metrics below them use the same rule on an uncorrected p-value, so their verdicts only describe the run.

In [ ]:
show.verdicts()

## 2 · Start-up

Time to ready runs from just before the runner asks for the side's pod until the server lists the model; time to first answer, until the first generated text of a streamed request comes back. Both are timed by the runner, the same way for every image, and the warm-up rounds (which pull the image and download the model) are left out. A side set to compileCache warm starts its scored rounds with the compile caches its warm-up round left; the table in section 5 shows each side's setting. The phases below come from the servers' own log lines, as each server words them.

In [ ]:
show.startup()

## 3 · Energy per token

Output tokens per joule at each concurrency level: the level's output tokens per second divided by the mean GPU power over the level's time window. The power is GPU-only power, the board power the GPU reports; the rest of the machine is not in it.

In [ ]:
show.energy()

## 4 · Scale-out

Runs in mode scaleout only. In each round one replica of the side starts and answers; GuideLLM then sends requests at a fixed rate (the surge) through a router that sends each request to the replica with the fewest requests in flight. At a fixed time after the surge's first request (the forced step) the runner starts a second replica of the same image, and the router sends it requests once it answers. The new replica's start is timed like the first one's; the surge's figures are GuideLLM's, against latency goals set before the first round.

In [ ]:
show.scale_out()

## 5 · What was compared, and how

The images as the node pulled them, the versions the servers reported, the GPU and the node, and the method declared before the first round. With A/A pairs in the plan, the last table shows how far two rounds of the same image land apart.

In [ ]:
show.provenance()